# Diffusion-NullFusion on Kaggle

This notebook downloads the Chikusei dataset, runs the Diffusion-NullFusion training with the `--large_unet` flag, and checks that the resulting PSNR exceeds the current SOTA (>= 49.14 dB).


In [ ]:
import json, os, pathlib, stat
kaggle_dir = pathlib.Path('/kaggle/working/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)
(kaggle_dir / 'kaggle.json').write_text(json.dumps({
    'username': 'amarnathmadaka',
    'key': '9ab5e3ea0d7d8fbf7cb90dd63b8d8835'
}))
os.chmod(str(kaggle_dir / 'kaggle.json'), stat.S_IRUSR | stat.S_IWUSR)
print('Kaggle credentials written')

In [ ]:
# Install Kaggle CLI (Kaggle kernels already have it, but ensure)
!pip install --quiet kaggle

In [ ]:
# Download Chikusei dataset
!kaggle datasets download -d mingliu123/chikusei -p /kaggle/working --unzip

In [ ]:
# Run the training script
!python ../../scripts/train_diffusion_nullfusion_chikusei.py \
    --root /kaggle/working/chikusei \
    --large_unet \
    --time_budget_h 8.5 \
    --epochs 1500 \
    --batch_size 1 \
    --patch 48

In [ ]:
import torch, pathlib
ckpt_path = pathlib.Path('diffusion_nullfusion') / 'best.pt'
if ckpt_path.exists():
    ck = torch.load(ckpt_path, map_location='cpu')
    psnr = ck.get('val', {}).get('psnr')
    print(f'Best PSNR from checkpoint: {psnr}')
    assert psnr is not None, 'PSNR not found in checkpoint'
    target = 49.14
    if psnr >= target:
        print('SOTA benchmark beaten!')
    else:
        print(f'Did not beat SOTA. PSNR {psnr:.2f} < {target}')
else:
    print('Checkpoint not found. Training may have failed.')